# Bellman Backup on 3-State MDP

Chapter 15: Bellman Equations and Operators · *Mathematics for AI and Machine Learning* by Fuheng Wu

Apply one-step Bellman expectation updates and inspect residuals state-by-state.

Companion code for the [interactive demo](https://math4ai.org/companion/chapter-15/bellman-backup/) on math4ai.org (sign-in required). Run the cell with Shift+Enter; PyTorch is preinstalled on Colab. Change the values at the top and run it again to explore.


In [1]:
import torch

torch.set_printoptions(precision=4)

# The demo's defaults: discount, self-transition bias, and the first state's value
gamma, b, v0 = 0.90, 0.65, 0.2
P = torch.tensor([[b, 1 - b, 0.0],
                  [0.2, 0.6, 0.2],
                  [0.0, 1 - b, b]], dtype=torch.float64)       # rows sum to 1
r = torch.tensor([0.2, 0.8, 1.4], dtype=torch.float64)
V = torch.tensor([v0, 1.0, -0.4], dtype=torch.float64)


def T(V):
    """Bellman expectation operator for a fixed policy: (T V)(s) = r(s) + gamma sum_s' P(s'|s) V(s')."""
    return r + gamma * P @ V


TV = T(V)
print("V      =", V.tolist())
print("T(V)   =", TV.tolist())
print("residual T(V) - V =", (TV - V).tolist(), " sup-norm =", (TV - V).abs().max().item())

# State 0 by hand: r0 + gamma (b V0 + (1 - b) V1)
print("state 0 by hand:", 0.2 + gamma * (b * v0 + (1 - b) * 1.0))

# ── Repeating the backup converges to V^pi, the fixed point T(V) = V ────────
V_pi = torch.linalg.solve(torch.eye(3, dtype=torch.float64) - gamma * P, r)
W = V.clone()
for k in range(1, 61):
    W = T(W)
    if k in (1, 2, 5, 10, 20, 40, 60):
        print(f"after {k:2d} backups: ||V_k - V^pi||_inf = {(W - V_pi).abs().max().item():.3e}")
print("V^pi =", V_pi.tolist(), " T(V^pi) == V^pi:", torch.allclose(T(V_pi), V_pi))

V      = [0.2, 1.0, -0.4]
T(V)   = [0.6320000000000001, 1.304, 1.4809999999999999]
residual T(V) - V = [0.4320000000000001, 0.30400000000000005, 1.8809999999999998]  sup-norm = 1.8809999999999998
state 0 by hand: 0.632
after  1 backups: ||V_k - V^pi||_inf = 7.965e+00
after  2 backups: ||V_k - V^pi||_inf = 6.769e+00
after  5 backups: ||V_k - V^pi||_inf = 4.600e+00
after 10 backups: ||V_k - V^pi||_inf = 2.654e+00
after 20 backups: ||V_k - V^pi||_inf = 9.224e-01
after 40 backups: ||V_k - V^pi||_inf = 1.121e-01
after 60 backups: ||V_k - V^pi||_inf = 1.363e-02
V^pi = [6.5542168674698855, 8.000000000000005, 9.445783132530126]  T(V^pi) == V^pi: True
